In [1]:
# install fastkaggle if not available
try: import fastkaggle
except ModuleNotFoundError:
    !pip install -Uq fastkaggle

from fastkaggle import *

这是[Road to the Top](https://www.kaggle.com/code/jhoward/first-steps-road-to-the-top-part-1)系列的第2部分，在本系列中，我展示了用于解决[Paddy Doctor](https://www.kaggle.com/competitions/paddy-disease-classification)竞赛的过程，最终取得了四次第一名的成绩。如果您还没有阅读，请先查看[第1部分](https://www.kaggle.com/code/jhoward/first-steps-road-to-the-top-part-1)。


## 加速进行


首先，我们将重复上次使用的步骤来访问数据，并确保所有最新的库都已安装：


In [2]:
comp = 'paddy-disease-classification'
path = setup_comp(comp, install='fastai "timm>=0.6.2.dev0"')
from fastai.vision.all import *
set_seed(42)

我上次注意到一个很大的问题：最初我是在家用电脑上创建的 notebook，我们创建的 resnet 每个 epoch 只需不到 20 秒。但在 Kaggle 上，每个 epoch 却需要超过 3 分钟！虽然 Kaggle 的 GPU 没有我家里的强，但这远远不足以解释如此巨大的速度差异。

我注意到 Kaggle 运行时，右上角的"GPU"指示器几乎是空的，而"CPU"指示器始终是满的。这强烈说明问题在于 Kaggle 的 notebook 在解码和缩放图像时受到了 CPU 瓶颈的制约。这在 CPU 性能较差的机器上是个常见问题——实际上，截至本文撰写时，Kaggle 只提供 2 个虚拟 CPU。

我们确实需要解决这个问题，因为我们需要能够更快速地进行迭代。我们可以直接将所有图像的高度和宽度缩小到原来的一半——这样像素数量减少为原来的四分之一，从而在训练小模型时性能大约能提升 4 倍。

幸运的是，fastai 提供了一个函数可以做到这一点，同时保持数据的文件夹结构不变：`resize_images`。


In [7]:
trn_path = Path('sml')

In [8]:
resize_images(path/'train_images', dest=trn_path, max_size=256, recurse=True)

这将给我们提供 192x256 像素的图像。让我们来看看：


In [11]:
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, seed=42,
    item_tfms=Resize((256,192)))

dls.show_batch(max_n=3)

在这个笔记本中，我们将尝试几种不同的架构和图像处理方法（item 变换和 batch 变换）。为了使这更容易，我们将把建模步骤整合到一个小函数中，我们可以将架构、item 变换和 batch 变换传递给它：


In [13]:
def train(arch, item, batch, epochs=5):
    dls = ImageDataLoaders.from_folder(trn_path, seed=42, valid_pct=0.2, item_tfms=item, batch_tfms=batch)
    learn = vision_learner(dls, arch, metrics=error_rate).to_fp16()
    learn.fine_tune(epochs, 0.01)
    return learn

我们的 `item_tfms` 已经将图像调整为较小的尺寸，所以这对我们模型的准确性影响应该不大，甚至毫无影响。让我们重新运行 resnet26d 来测试一下。


In [12]:
learn = train('resnet26d', item=Resize(192),
              batch=aug_transforms(size=128, min_scale=0.75))

这在速度上是一个很大的提升，而且准确性看起来也不错。


## ConvNeXt 模型


我注意到 Kaggle 中的 GPU 使用率条仍然几乎为空，所以我们仍然受到 CPU 的限制。这意味着我们应该能够使用更强大的模型，而对速度几乎没有影响。让我们再看看[最适合微调的视觉模型](https://www.kaggle.com/code/jhoward/the-best-vision-models-for-fine-tuning)中的选项。`convnext_small` 在性能/精度权衡得分中名列前茅，让我们来试试吧！


In [14]:
arch = 'convnext_small_in22k'

In [15]:
learn = train(arch, item=Resize(192, method='squish'),
              batch=aug_transforms(size=128, min_scale=0.75))

哇，我们的错误率减半了！这是个很棒的结果。而且，正如预期的那样，速度几乎没有提升。这看起来是一个非常适合迭代的模型。


## 预处理实验


那么，我们首先尝试什么呢？一个可能产生影响的因素是：我们是通过改变纵横比将矩形图像"压缩"成正方形，还是从中随机裁剪出一个正方形，抑或是在边缘添加黑色填充使其成为正方形。在之前的版本中，我们使用了"压缩"。现在让我们尝试"裁剪"，这也是 fastai 的默认方式：


In [17]:
learn = train(arch, item=Resize(192),
              batch=aug_transforms(size=128, min_scale=0.75))

这似乎没有太大区别……

我们还可以尝试填充（padding），它会保留所有原始图像而不进行任何变换——效果如下：


In [16]:
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, seed=42,
    item_tfms=Resize(192, method=ResizeMethod.Pad, pad_mode=PadMode.Zeros))
dls.show_batch(max_n=3)

In [18]:
learn = train(arch, item=Resize((256,192), method=ResizeMethod.Pad, pad_mode=PadMode.Zeros),
      batch=aug_transforms(size=(171,128), min_scale=0.75))

这看起来是一个相当不错的改进。


## 测试时数据增强


为了进一步提升预测效果，我们可以尝试[测试时数据增强](https://nbviewer.org/github/fastai/fastbook/blob/master/07_sizing_and_tta.ipynb#Test-Time-Augmentation)（TTA）。[我们的书](https://www.amazon.com/Deep-Learning-Coders-fastai-PyTorch/dp/1492045527)对此的定义如下：

> *在推理或验证阶段，对每张图像通过数据增强生成多个版本，再对各增强版本的预测结果取平均值或最大值。*

在尝试 TTA 之前，我们先来看看如何在不使用 TTA 的情况下检查模型的预测结果和错误率：


In [19]:
valid = learn.dls.valid
preds,targs = learn.get_preds(dl=valid)

In [20]:
error_rate(preds, targs)

这与我们在上面训练结束时看到的错误率相同，所以我们知道操作是正确的。

以下是我们的数据增强所做的事情——如果你仔细观察，可以看到每张图像都有些许明暗变化，有时还会被翻转、缩放、旋转、扭曲，以及/或者变焦：


In [21]:
learn.dls.train.show_batch(max_n=6, unique=True)

如果我们调用 `tta()`，我们将得到对每张图像的多个不同增强版本（以及未增强的原始图像）所做预测的平均值：


In [22]:
tta_preds,_ = learn.tta(dl=valid)

让我们检查一下这个的错误率：


In [23]:
error_rate(tta_preds, targs)

这真是一个巨大的进步！我们肯定会在任何提交中使用它！


## 扩大规模


现在我们已经有了一个相当不错的模型和预处理方法，让我们将其扩展到更大的图像和更多的训练轮次。我们将把路径切换回原始未缩放的图像，并使用迄今为止最佳设置运行 12 个轮次，同时使用更大的最终增强图像：


In [24]:
trn_path = path/'train_images'

In [26]:
learn = train(arch, epochs=12,
              item=Resize((480, 360), method=ResizeMethod.Pad, pad_mode=PadMode.Zeros),
              batch=aug_transforms(size=(256,192), min_scale=0.75))

这大约是我们之前最佳模型准确率的两倍——让我们也看看它在 TTA 下的表现：


In [34]:
tta_preds,targs = learn.tta(dl=learn.dls.valid)
error_rate(tta_preds, targs)

再一次，我们从TTA中获得了巨大的提升。在我看来，这是最被低估的深度学习技巧之一！（我不确定是否有其他框架能让它变得如此简单，所以也许这就是原因之一……）


## 提交


我们现在准备好整理我们的 Kaggle 提交了。首先，我们将像在上一个笔记本中那样获取测试集：


In [27]:
tst_files = get_image_files(path/'test_images').sorted()
tst_dl = learn.dls.test_dl(tst_files)

接下来，对测试集进行TTA：


In [28]:
preds,_ = learn.tta(dl=tst_dl)

我们需要每行中最大概率预测的索引，因为那就是预测疾病的索引。PyTorch 中的 `argmax` 正好可以做到这一点：


In [29]:
idxs = preds.argmax(dim=1)

现在我们需要在 `vocab` 中查找这些索引。上次我们使用 pandas 来完成这个操作，不过从那以后我意识到还有一种更简单的方法！：


In [30]:
vocab = np.array(learn.dls.vocab)
results = pd.Series(vocab[idxs], name="idxs")

In [31]:
ss = pd.read_csv(path/'sample_submission.csv')
ss['label'] = results
ss.to_csv('subm.csv', index=False)
!head subm.csv

In [ ]:
if not iskaggle:
    from kaggle import api
    api.competition_submit_cli('subm.csv', 'convnext small 256x192 12 epochs tta', comp)

这个得分为 0.9827，远在比赛前 25% 之内——这是一个很大的进步，而且我们仍然只使用了一个小型模型！


In [ ]:
# This is what I use to push my notebook from my home PC to Kaggle

if not iskaggle:
    push_notebook('jhoward', 'small-models-road-to-the-top-part-2',
                  title='Small models: Road to the Top, Part 2',
                  file='small-models-road-to-the-top-part-2.ipynb',
                  competition=comp, private=True, gpu=True)

## 结论


今天我们取得了重要进展，尽管只使用了一个单一模型，即便在 Kaggle 配置相对较低的机器上，训练时间也不超过 20 分钟。下次我们将尝试扩展到更大的模型，并进行一些集成学习。

如果您觉得这个 notebook 对您有帮助，请记得点击顶部的小上箭头为它点赞——我喜欢了解大家是否觉得我的工作有价值，这也能帮助更多人发现它。如果您有任何问题或意见，欢迎在下方留言——我会阅读每一条评论！


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文档已使用 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言的原始文档应被视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
